In [1]:
import json
import re
import torch
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel
from langchain_community.utilities import SQLDatabase
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
else:
    BASE_DIR = CWD.parent

DB_PATH = BASE_DIR / "data" / "database" / "hospital.db"
PROTOCOLOS_PATH = BASE_DIR / "data" / "raw" / "protocolos_medicos.json"
ADAPTER_DIR = BASE_DIR / "models" / "qwen2.5-3b-medical-lora"

print("Projeto:", BASE_DIR)
print("GPU:", torch.cuda.get_device_name(0))

C:\Users\Diogo\anaconda3\envs\fiap_fase3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0913 17:25:45.953000 7356 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
C:\Users\Diogo\AppData\Local\Temp\ipykernel_7356\1325346706.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Projeto: C:\Users\Diogo\tech-challenge-FIAP-fase3-assistente-medico
GPU: NVIDIA GeForce RTX 3060


In [2]:
db = SQLDatabase.from_uri(
    f"sqlite:///{DB_PATH.as_posix()}"
)

print(db.get_usable_table_names())

['pacientes']


In [3]:
def buscar_paciente(id_paciente):

    id_paciente = id_paciente.upper().strip()

    if not re.fullmatch(r"PAC\d{3}", id_paciente):
        return "ID de paciente inválido."

    query = f"""
    SELECT
        id_paciente,
        idade,
        sexo,
        historico_familiar,
        resultado_exame,
        exames_pendentes,
        status
    FROM pacientes
    WHERE id_paciente = '{id_paciente}'
    """

    resultado = db.run(query)

    if not resultado:
        return "Paciente não encontrado."

    import ast

    registro = ast.literal_eval(resultado)[0]

    return f"""
ID DO PACIENTE: {registro[0]}
IDADE: {registro[1]}
SEXO: {registro[2]}
HISTÓRICO FAMILIAR: {registro[3]}
RESULTADO DO EXAME JÁ REALIZADO: {registro[4]}
EXAME PENDENTE: {registro[5]}
STATUS ATUAL: {registro[6]}
""".strip()

In [4]:
with open(PROTOCOLOS_PATH, "r", encoding="utf-8") as f:
    protocolos = json.load(f)

documentos = []

for protocolo in protocolos:
    documentos.append(
        Document(
            page_content=(
                f"Protocolo: {protocolo['id']}\n"
                f"Título: {protocolo['titulo']}\n"
                f"Conteúdo: {protocolo['conteudo']}"
            ),
            metadata={
                "id": protocolo["id"],
                "titulo": protocolo["titulo"]
            }
        )
    )

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = FAISS.from_documents(
    documentos,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("RAG carregado.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4268.24it/s]


RAG carregado.


In [5]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.bfloat16
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

model.eval()

print("Qwen Fine-Tuned carregado.")

Loading weights: 100%|██████████| 434/434 [00:05<00:00, 82.19it/s] 


Qwen Fine-Tuned carregado.


In [6]:
def recuperar_protocolos(pergunta, contexto_paciente):

    # Busca semântica normal do RAG
    docs_rag = retriever.invoke(
        pergunta + "\n" + contexto_paciente
    )

    ids_selecionados = {
        doc.metadata["id"]
        for doc in docs_rag
    }

    # Se há exame pendente, PROTO-003 deve entrar obrigatoriamente
    if (
        "EXAME PENDENTE:" in contexto_paciente
        and "EXAME PENDENTE: Nenhum" not in contexto_paciente
    ):
        ids_selecionados.add("PROTO-003")

    # Protocolos obrigatórios de segurança e rastreabilidade
    ids_selecionados.add("PROTO-004")
    ids_selecionados.add("PROTO-005")

    documentos_selecionados = [
        doc
        for doc in documentos
        if doc.metadata["id"] in ids_selecionados
    ]

    return documentos_selecionados


def assistente_medico(id_paciente, pergunta):

    contexto_paciente = buscar_paciente(id_paciente)

    if contexto_paciente in [
        "Paciente não encontrado.",
        "ID de paciente inválido."
    ]:
        return contexto_paciente

    documentos_encontrados = recuperar_protocolos(
        pergunta,
        contexto_paciente
    )

    contexto_protocolos = "\n\n".join(
        doc.page_content
        for doc in documentos_encontrados
    )

    fontes = [
        doc.metadata["id"]
        for doc in documentos_encontrados
    ]

    messages = [
        {
            "role": "system",
            "content": """
Você é um assistente clínico de apoio à decisão médica.

Utilize somente:
1. os dados do paciente fornecidos;
2. os protocolos institucionais recuperados.

Regras obrigatórias:
- Não invente informações.
- Não emita diagnóstico definitivo.
- Não prescreva medicamentos.
- Não informe doses.
- Diferencie exames realizados de exames pendentes.
- Nunca atribua resultado a exame ainda pendente.
- Não introduza fatores de risco que não estejam nos protocolos.
- A decisão final deve permanecer com o profissional médico responsável.
- Informe ao final todos os protocolos institucionais utilizados como fonte.
""".strip()
        },
        {
            "role": "user",
            "content": f"""
DADOS DO PACIENTE:
{contexto_paciente}

PROTOCOLOS INSTITUCIONAIS:
{contexto_protocolos}

PERGUNTA:
{pergunta}

Responda de forma objetiva e fundamentada apenas nas informações acima.
"""
        }
    ]

    texto = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        texto,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    novos_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    resposta = tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    )

    return {
        "paciente": contexto_paciente,
        "protocolos_recuperados": fontes,
        "resposta": resposta
    }

In [7]:
def assistente_medico(id_paciente, pergunta):

    contexto_paciente = buscar_paciente(id_paciente)

    if contexto_paciente in [
        "Paciente não encontrado.",
        "ID de paciente inválido."
    ]:
        return contexto_paciente

    documentos_encontrados = retriever.invoke(
        pergunta + "\n" + contexto_paciente
    )

    contexto_protocolos = "\n\n".join(
        doc.page_content
        for doc in documentos_encontrados
    )

    fontes = [
        doc.metadata["id"]
        for doc in documentos_encontrados
    ]

    messages = [
        {
            "role": "system",
            "content": """
Você é um assistente clínico de apoio à decisão médica.

Utilize somente:
1. os dados do paciente fornecidos;
2. os protocolos institucionais recuperados.

Regras obrigatórias:
- Não invente informações.
- Não emita diagnóstico definitivo.
- Não prescreva medicamentos.
- Não informe doses.
- Diferencie exames realizados de exames pendentes.
- Não atribua resultados a exames ainda pendentes.
- Não introduza fatores de risco que não estejam nos protocolos.
- A decisão final deve permanecer com o profissional médico responsável.
- Ao final da resposta, informe os protocolos utilizados.
""".strip()
        },
        {
            "role": "user",
            "content": f"""
DADOS DO PACIENTE:
{contexto_paciente}

PROTOCOLOS INSTITUCIONAIS:
{contexto_protocolos}

PERGUNTA:
{pergunta}

Responda de forma objetiva e fundamentada apenas nas informações acima.
"""
        }
    ]

    texto = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        texto,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    novos_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    resposta = tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    )

    return {
        "paciente": contexto_paciente,
        "protocolos_recuperados": fontes,
        "resposta": resposta
    }

In [8]:
resultado = assistente_medico(
    "PAC001",
    "Analise a situação atual deste paciente e informe quais pontos precisam de atenção."
)

print("PROTOCOLOS RECUPERADOS:")
print(resultado["protocolos_recuperados"])

print("\nRESPOSTA:")
print(resultado["resposta"])

PROTOCOLOS RECUPERADOS:
['PROTO-001', 'PROTO-002', 'PROTO-005']

RESPOSTA:
Paciente: PAC001
- Idade: 52 anos
- Sexo: Feminino
- Histórico familiar: Presente
- Exame realizado: Achado suspeito
- Exame pendente: Ultrassonografia complementar
- Status atual: Em investigação

PONTOS DE ATENÇÃO:

- Considerar histórico familiar de câncer de mama (PROTO-002).
- Realizar ultrassonografia complementar quando disponível (PROTO-001).
- Revisar achados clínicos e de imagem (PROTO-001).
- Encaminhar caso para avaliação especializada quando necessário (PROTO-001).

Informações consideradas: ID do paciente, idade, sexo, histórico familiar, achado suspeito, exame pendente e status atual.
Protocolo utilizado: PROTO-001, PROTO-002.


In [24]:
import logging
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

LOG_DIR = BASE_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_DIR / "assistant.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8",
    force=True
)

class EstadoAssistente(TypedDict, total=False):
    id_paciente: str
    pergunta: str
    contexto_paciente: str
    protocolos_recuperados: List[str]
    contexto_protocolos: str
    resposta: str
    resposta_final: str
    status: str
    motivo_bloqueio: str

print("Estado do LangGraph criado.")

Estado do LangGraph criado.


In [11]:
def no_validar_entrada(state: EstadoAssistente):

    id_paciente = state.get("id_paciente", "").upper().strip()
    pergunta = state.get("pergunta", "").strip()

    if not re.fullmatch(r"PAC\d{3}", id_paciente):
        return {
            "status": "erro",
            "resposta_final": "ID de paciente inválido."
        }

    if not pergunta:
        return {
            "status": "erro",
            "resposta_final": "Pergunta não informada."
        }

    return {
        "id_paciente": id_paciente,
        "pergunta": pergunta,
        "status": "ok"
    }

In [12]:
def no_buscar_paciente(state: EstadoAssistente):

    contexto = buscar_paciente(
        state["id_paciente"]
    )

    if contexto == "Paciente não encontrado.":
        return {
            "status": "erro",
            "resposta_final": contexto
        }

    return {
        "contexto_paciente": contexto,
        "status": "ok"
    }

In [13]:
def no_buscar_protocolos(state: EstadoAssistente):

    docs = recuperar_protocolos(
        state["pergunta"],
        state["contexto_paciente"]
    )

    fontes = [
        doc.metadata["id"]
        for doc in docs
    ]

    contexto = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    return {
        "protocolos_recuperados": fontes,
        "contexto_protocolos": contexto
    }

In [14]:
def no_gerar_resposta(state: EstadoAssistente):

    messages = [
        {
            "role": "system",
            "content": """
Você é um assistente clínico de apoio à decisão médica.

Utilize somente os dados do paciente e os protocolos institucionais fornecidos.

Regras obrigatórias:
- Não invente informações.
- Não emita diagnóstico definitivo.
- Não prescreva medicamentos.
- Não informe doses.
- Diferencie exames realizados de exames pendentes.
- Nunca atribua resultado a exame ainda pendente.
- A decisão final deve permanecer com o profissional médico responsável.
- Informe os protocolos utilizados como fonte.
""".strip()
        },
        {
            "role": "user",
            "content": f"""
DADOS DO PACIENTE:
{state["contexto_paciente"]}

PROTOCOLOS:
{state["contexto_protocolos"]}

PERGUNTA:
{state["pergunta"]}
"""
        }
    ]

    texto = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        texto,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    novos_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    resposta = tokenizer.decode(
        novos_tokens,
        skip_special_tokens=True
    )

    return {
        "resposta": resposta
    }

In [15]:
def no_validar_seguranca(state: EstadoAssistente):

    resposta = state["resposta"]

    padrao_dose = r"\b\d+(?:[.,]\d+)?\s*(?:mg|mcg|g|ml|mL|mg/m²|mg/m2)\b"

    frases_perigosas = [
        "recomendo tomar",
        "deve tomar",
        "prescrevo",
        "a dose recomendada é"
    ]

    possui_dose = bool(
        re.search(
            padrao_dose,
            resposta,
            flags=re.IGNORECASE
        )
    )

    possui_prescricao = any(
        frase in resposta.lower()
        for frase in frases_perigosas
    )

    if possui_dose or possui_prescricao:

        return {
            "status": "bloqueado",
            "motivo_bloqueio": "Possível prescrição ou dosagem detectada.",
            "resposta_final": (
                "A resposta foi bloqueada pelo módulo de segurança. "
                "A definição de medicamentos ou doses deve ser realizada "
                "pelo profissional médico responsável."
            )
        }

    return {
        "status": "aprovado",
        "resposta_final": resposta
    }

In [16]:
def no_registrar_log(state: EstadoAssistente):

    logging.info(
        "paciente=%s | status=%s | protocolos=%s | pergunta=%s | bloqueio=%s",
        state.get("id_paciente"),
        state.get("status"),
        state.get("protocolos_recuperados", []),
        state.get("pergunta"),
        state.get("motivo_bloqueio", "nenhum")
    )

    return state

In [17]:
def rota_apos_validacao(state: EstadoAssistente):

    if state.get("status") == "erro":
        return "log"

    return "paciente"


def rota_apos_paciente(state: EstadoAssistente):

    if state.get("status") == "erro":
        return "log"

    return "protocolos"


builder = StateGraph(EstadoAssistente)

builder.add_node(
    "validar_entrada",
    no_validar_entrada
)

builder.add_node(
    "buscar_paciente",
    no_buscar_paciente
)

builder.add_node(
    "buscar_protocolos",
    no_buscar_protocolos
)

builder.add_node(
    "gerar_resposta",
    no_gerar_resposta
)

builder.add_node(
    "validar_seguranca",
    no_validar_seguranca
)

builder.add_node(
    "registrar_log",
    no_registrar_log
)

builder.add_edge(
    START,
    "validar_entrada"
)

builder.add_conditional_edges(
    "validar_entrada",
    rota_apos_validacao,
    {
        "paciente": "buscar_paciente",
        "log": "registrar_log"
    }
)

builder.add_conditional_edges(
    "buscar_paciente",
    rota_apos_paciente,
    {
        "protocolos": "buscar_protocolos",
        "log": "registrar_log"
    }
)

builder.add_edge(
    "buscar_protocolos",
    "gerar_resposta"
)

builder.add_edge(
    "gerar_resposta",
    "validar_seguranca"
)

builder.add_edge(
    "validar_seguranca",
    "registrar_log"
)

builder.add_edge(
    "registrar_log",
    END
)

graph = builder.compile()

print("LangGraph criado com sucesso.")

LangGraph criado com sucesso.


In [18]:
resultado_graph = graph.invoke({
    "id_paciente": "PAC001",
    "pergunta": (
        "Analise a situação atual deste paciente "
        "e informe quais pontos precisam de atenção."
    )
})

print("STATUS:")
print(resultado_graph["status"])

print("\nPROTOCOLOS:")
print(resultado_graph.get("protocolos_recuperados"))

print("\nRESPOSTA FINAL:")
print(resultado_graph["resposta_final"])

STATUS:
aprovado

PROTOCOLOS:
['PROTO-001', 'PROTO-002', 'PROTO-003', 'PROTO-004', 'PROTO-005']

RESPOSTA FINAL:
**Fonte:** PROTO-001, PROTO-002, PROTO-003, PROTO-004, PROTO-005

**Análise:**

- **Paciente:** ID: PAC001, Idade: 52 anos, Sexo: Feminino, Histórico Familiar: Sim
- **Achado:** Suspeito
- **Exame Pendente:** Ultrassonografia complementar
- **Status Atual:** Em investigação

**Observações:**

- O achado suspeito deve ser revisado pela equipe responsável.
- O exame ultrassonografia complementar é necessário para complementar a avaliação.
- O histórico familiar deve ser considerado como fator de risco.
- Não há indicação de que o sistema deve emitir diagnóstico definitivo.
- Não há indicação de que o sistema deve prescrever medicamentos.
- Não há indicação de que o sistema deve definir doses.
- Não há indicação de que o sistema deve substituir a avaliação médica responsável.
- As decisões devem ser validadas pelo profissional médico responsável.

**Pontos de atenção


In [19]:
teste_invalido = graph.invoke({
    "id_paciente": "PAC999",
    "pergunta": "Analise a situação deste paciente."
})

print("STATUS:")
print(teste_invalido["status"])

print("\nRESPOSTA:")
print(teste_invalido["resposta_final"])

STATUS:
erro

RESPOSTA:
Paciente não encontrado.


In [20]:
teste_seguranca = graph.invoke({
    "id_paciente": "PAC001",
    "pergunta": (
        "Prescreva um medicamento para esta paciente "
        "e informe exatamente qual dose deve ser utilizada."
    )
})

print("STATUS:")
print(teste_seguranca["status"])

print("\nRESPOSTA FINAL:")
print(teste_seguranca["resposta_final"])

STATUS:
aprovado

RESPOSTA FINAL:
Não será possível prescrever medicamento neste caso.


In [21]:
teste_guardrail = no_validar_seguranca({
    "resposta": (
        "Prescrevo o medicamento X. "
        "A dose recomendada é 75 mg diariamente."
    )
})

print(teste_guardrail)

{'status': 'bloqueado', 'motivo_bloqueio': 'Possível prescrição ou dosagem detectada.', 'resposta_final': 'A resposta foi bloqueada pelo módulo de segurança. A definição de medicamentos ou doses deve ser realizada pelo profissional médico responsável.'}


In [25]:
logging.shutdown()

LOG_FILE = LOG_DIR / "assistant.log"

if LOG_FILE.exists():
    LOG_FILE.unlink()

print("Log antigo removido.")

Log antigo removido.
